# Domain-Adaptive Image Colorization (MoE Pipeline)
This notebook runs the end-to-end Mixture-of-Experts (MoE) Image Colorization pipeline:
- **Person A:** Grounded-SAM object detection, segmentation, and domain routing (`people`, `nature`, `urban`, `animals`, `fallback`)
- **Person B:** Region extraction with contextual padding, DDColor expert colorization, soft Gaussian mask blending, and RGB reconstruction.

### Kaggle Setup Requirements (Right Sidebar):
1. **Accelerator:** GPU (T4 x2 or P100 recommended)
2. **Internet:** **ON** (Required for `pip install` and downloading model checkpoints)


In [ ]:
# Cell 1: Clone repository (if running in a fresh Kaggle session)
import os
if not os.path.exists("colorize"):
    !git clone -b B https://github.com/zaeem886/domain-adaptive-colorization.git
    %cd domain-adaptive-colorization
else:
    print("Already in project directory:", os.getcwd())
    !git pull origin B


In [ ]:
# Cell 2: Install required packages
!pip install -q opencv-python segment-anything timm transformers einops
# Note: --no-build-isolation fixes PEP 517 build errors for basicsr
!pip install -q basicsr --no-build-isolation || echo "basicsr wheel build skipped"


In [ ]:
# Cell 3: Verify all components with unit tests (Person A & Person B)
!PYTHONPATH=. python tests/run_tests.py


In [ ]:
# Cell 4: Run manual test to demonstrate soft blending vs hard pasting
!python manual_test.py pics/dog.jpeg


In [ ]:
# Cell 5: Display 4-panel visual comparison
# Panels: [1] Grayscale input | [2] Domain Map | [3] Hard Paste (Blend OFF) | [4] Soft Blended (Blend ON)
from IPython.display import Image, display
display(Image(filename="manual_test_output/05_comparison_side_by_side.png"))


In [ ]:
# Cell 6: Download SAM ViT-B checkpoint for Grounded-SAM
import os
import urllib.request

SAM_CHECKPOINT = "sam_vit_b.pth"
SAM_URL = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"

if not os.path.exists(SAM_CHECKPOINT):
    print("Downloading SAM checkpoint (~375MB)...")
    urllib.request.urlretrieve(SAM_URL, SAM_CHECKPOINT)
    print("Downloaded successfully!")
else:
    print("SAM checkpoint already present.")


In [ ]:
# Cell 7: Run Grounded-SAM (Person A) on GPU
import torch
from colorize.config import DecompConfig
from colorize.decomposition import GroundedSAM, load_gray

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

decomp_cfg = DecompConfig(sam_checkpoint=SAM_CHECKPOINT)
grounded_sam = GroundedSAM(cfg=decomp_cfg, device=device)

gray_img = load_gray("pics/dog.jpeg", max_side=600)
decomp = grounded_sam(gray_img)

print(f"[Person A] Routed {len(decomp.regions)} regions.")
print(f"[Person A] Domain coverage: {decomp.coverage * 100:.1f}%")
print("[Person A] Domain shares:", decomp.domain_share)


In [ ]:
# Cell 8: (Optional) Full Colorization Pipeline with Expert Checkpoints
# Populate checkpoints with paths to your trained domain weights:
# from colorize.pipeline import ColorizationPipeline
# from colorize.colorization_config import ColorizationConfig
# 
# checkpoints = {
#     "people": "checkpoints/ddcolor_people.pth",
#     "nature": "checkpoints/ddcolor_nature.pth",
#     "urban": "checkpoints/ddcolor_urban.pth",
#     "animals": "checkpoints/ddcolor_animals.pth",
#     "fallback": "checkpoints/ddcolor_fallback.pth",
# }
# pipeline = ColorizationPipeline(registry=checkpoints, cfg=ColorizationConfig())
